In [26]:
import gspread
from google.oauth2.service_account import Credentials
from langchain_core.tools import tool

# Define the scope
SCOPES = [
    'https://www.googleapis.com/auth/spreadsheets',
    'https://www.googleapis.com/auth/drive'
]

# Authorize gspread
credentials = Credentials.from_service_account_file("../credentials.json", scopes=SCOPES)
gc = gspread.authorize(credentials)

@tool
def read_sheet(spreadsheet_id: str, sheet_name: str) -> list:
    """Reads all data from a worksheet given its ID."""
    spreadsheet = gc.open_by_key(spreadsheet_id)
    worksheet = spreadsheet.worksheet(sheet_name)
    return worksheet.get_all_records()

@tool
def update_cell(spreadsheet_id: str, sheet_name: str, a1_notation, value: str) -> str:
    """Updates a specific cell identified by A1 notation."""
    spreadsheet = gc.open_by_key(spreadsheet_id)
    worksheet = spreadsheet.worksheet(sheet_name)
    worksheet.update_acell(a1_notation, value)
    return "SUCCESS"

from datetime import datetime
import re
VALID_CHANGE_TYPES = {"cancelled", "room_change", "substitution", "rescheduled"}
def normalise_class_id(class_id: str) -> str:
    return re.sub(r"^(\d+)([A-Z])([A-Z]{2,}(?:/[A-Z]+)?)$", r"\1\2 \3", class_id.strip())
@tool
def add_variation(
    spreadsheet_id: str,
    date: str,
    class_id: str,
    change_type: str,
    detail: str,
    source_email: str
) -> str:
    """
    Adds a row to the 'variations' tab recording a schedule change from an email.

    Args:
        spreadsheet_id: The Google Sheet ID
        date: The date of the variation in YYYY-MM-DD format
        class_id: The class affected e.g. '3D TEC'
        change_type: One of 'cancelled', 'room_change', 'substitution', 'rescheduled'
        detail: Plain English description of the change
        source_email: The filename of the source .eml file e.g. 'giovanni_march.eml'
    """
    # Validate change_type
    if change_type not in VALID_CHANGE_TYPES:
        return (
            f"Invalid change_type '{change_type}'. "
            f"Must be one of: {', '.join(sorted(VALID_CHANGE_TYPES))}"
        )

    # Validate date format
    try:
        datetime.strptime(date, "%Y-%m-%d")
    except ValueError:
        return f"Invalid date format '{date}'. Must be YYYY-MM-DD."

    # Validate class_id exists in classes tab
    spreadsheet = gc.open_by_key(spreadsheet_id)
    classes_ws = spreadsheet.worksheet("classes")
    class_records = classes_ws.get_all_records()
    normalised_input = normalise_class_id(class_id)
    known_ids = {normalise_class_id(r["class_id"]) for r in class_records}
    if normalised_input not in known_ids:
        return (
            f"Warning: class_id '{class_id}' (normalised: '{normalised_input}') "
            f"not found in 'classes' tab. Row was NOT written."
        )
    # known_ids = {r["class_id"] for r in class_records}
    # if class_id.strip() not in known_ids:
    #     return (
    #         f"Warning: class_id '{class_id}' not found in 'classes' tab. "
    #         f"Row was NOT written. Verify the class ID and retry."
    #     )

    # Write to variations tab
    variations_ws = spreadsheet.worksheet("variations")
    variations_ws.append_row([date, normalised_input, change_type, detail, source_email])

    return f"Variation added: {class_id} on {date} — {change_type} ({detail})"
@tool
def find_row(spreadsheet_id: str, sheet_name: str, query: str) -> list:
    """Finds the first row that contains the query string."""
    spreadsheet = gc.open_by_key(spreadsheet_id)
    worksheet = spreadsheet.worksheet(sheet_name)
    all_records = worksheet.get_all_records()
    for record in all_records:
        if query in str(record.values()):
            return [record]
    return []

In [27]:
file_path = "parsed_eml.txt"
with open(file_path, 'r') as file:
    email_content = file.read()

In [ ]:
# from IPython.display import display, Markdown, Image
# from typing import Annotated, TypedDict
# from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage, RemoveMessage
# from langchain_openai import ChatOpenAI
# from langgraph.graph import StateGraph, END, START
# from langgraph.prebuilt import ToolNode
# from langgraph.graph import MessagesState
# from langchain_core.runnables import RunnableConfig
# from langgraph.checkpoint.memory import MemorySaver
# from langgraph.graph.message import add_messages
# sys_msg = SystemMessage(content="You are a helpful bilingual Italian-English assistant whose job is to interpret Emails in parsed_eml.txt in both Italian and English, and add data to a Google Sheet tab. The id of the Google Sheet is 1nUhN2vGokOp2iGCkbGKT6aJT4_Ufzn6u-riS-ML_hyE. You have access to the following tools: read_sheet, update_cell, add_row, find_row. Use these tools to answer questions about the sheet." \
# "The Google Sheet has the following tabs: 'classes', 'curriculum', 'progress', 'assignments', 'schedule', and 'variations'. You will work in the 'variations' tab. The 'variations' tab has the following columns: 'date', 'class_id', 'change_type','detail', 'source_email'. The 'change_type' column has the following dropdown choices: 'cancelled', 'room_change', 'substitution', 'rescheduled'. The human message will contain the email content to be interpreted.Add data to the 'variations' tab from your interpretation of the email content." \
# )
# tools = [read_sheet, update_cell, add_variation, find_row]

# model = ChatOpenAI(model="gpt-4o", temperature=0)
# model_with_tools = model.bind_tools(tools)

# response = model_with_tools.invoke([sys_msg, HumanMessage(content=f"Add data to the 'variations' tab of the Google Sheet from your interpretation of the following email content: {email_content}")])




In [28]:
from langgraph.graph import StateGraph, END, START
from IPython.display import display, Markdown, Image
from typing import Annotated, TypedDict
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage, RemoveMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END, START
from langgraph.prebuilt import ToolNode
from langgraph.graph import MessagesState
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.message import add_messages
sys_msg = SystemMessage(content="You are a helpful bilingual Italian-English assistant whose job is to interpret Emails in parsed_eml.txt in both Italian and English, and add data to a Google Sheet tab. The id of the Google Sheet is 1nUhN2vGokOp2iGCkbGKT6aJT4_Ufzn6u-riS-ML_hyE. You have access to the following tools: read_sheet, update_cell, add_row, find_row. Use these tools to answer questions about the sheet." \
"The Google Sheet has the following tabs: 'classes', 'curriculum', 'progress', 'assignments', 'schedule', and 'variations'. You will work in the 'variations' tab. The 'variations' tab has the following columns: 'date', 'class_id', 'change_type','detail', 'source_email'. The 'change_type' column has the following dropdown choices: 'cancelled', 'room_change', 'substitution', 'rescheduled'. The human message will contain the email content to be interpreted.Add data to the 'variations' tab from your interpretation of the email content." \
)
tools = [read_sheet, update_cell, add_variation, find_row]
model = ChatOpenAI(model="gpt-4o", temperature=0)
model_with_tools = model.bind_tools(tools)
def chatbot(state):
    return {
        "messages": [
            model_with_tools.invoke(state["messages"])
        ]
    }

def should_continue(state):
    last = state["messages"][-1]
    if last.tool_calls:
        return "tools"
    return END

builder = StateGraph(MessagesState)
builder.add_edge(START, "chatbot")
builder.add_node("chatbot", chatbot)
builder.add_node("tools", ToolNode(tools))

builder.add_conditional_edges(
    "chatbot",
    should_continue,
    {
        "tools": "tools",
        END: END,
    },
)

builder.add_edge("tools", "chatbot")

graph = builder.compile()

In [29]:
graph.invoke({
    "messages": [
        sys_msg,
        HumanMessage(content=email_content)
    ]
})

{'messages': [SystemMessage(content="You are a helpful bilingual Italian-English assistant whose job is to interpret Emails in parsed_eml.txt in both Italian and English, and add data to a Google Sheet tab. The id of the Google Sheet is 1nUhN2vGokOp2iGCkbGKT6aJT4_Ufzn6u-riS-ML_hyE. You have access to the following tools: read_sheet, update_cell, add_row, find_row. Use these tools to answer questions about the sheet.The Google Sheet has the following tabs: 'classes', 'curriculum', 'progress', 'assignments', 'schedule', and 'variations'. You will work in the 'variations' tab. The 'variations' tab has the following columns: 'date', 'class_id', 'change_type','detail', 'source_email'. The 'change_type' column has the following dropdown choices: 'cancelled', 'room_change', 'substitution', 'rescheduled'. The human message will contain the email content to be interpreted.Add data to the 'variations' tab from your interpretation of the email content.", additional_kwargs={}, response_metadata={}